In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [2]:
nav_df = pd.read_csv('../Data/Processed/nav_history_clean.csv')
nav_df.head()

,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [3]:
nav_df.dtypes

date             str
amfi_code      int64
nav          float64
dtype: object

In [4]:
nav_df['date'] = pd.to_datetime(nav_df['date'])
nav_df = nav_df.sort_values(['amfi_code', 'date']).reset_index(drop=True)
nav_df.head()

,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [5]:
print(nav_df['amfi_code'].nunique())
print(nav_df.shape)

40
(64320, 3)


In [6]:
nav_df['daily_return'] = nav_df.groupby('amfi_code')['nav'].pct_change()
nav_df.head(10)

,date,amfi_code,nav,daily_return
0,2022-01-03,100016,520.4608,NaN
1,2022-01-04,100016,515.0971,-0.010306
2,2022-01-05,100016,521.7239,0.012865
3,2022-01-06,100016,515.7880,-0.011377
4,2022-01-07,100016,515.1639,-0.001210
5,2022-01-08,100016,515.1639,0.000000
6,2022-01-09,100016,515.1639,0.000000
7,2022-01-10,100016,510.7136,-0.008639
8,2022-01-11,100016,513.5542,0.005562
9,2022-01-12,100016,512.3195,-0.002404


In [7]:
nav_df['daily_return'].describe()

count    64280.000000
mean         0.000451
std          0.008706
min         -0.058102
25%         -0.002092
50%          0.000000
75%          0.003233
max          0.064713
Name: daily_return, dtype: float64

In [8]:
latest_date = nav_df['date'].max()
print(latest_date)

2026-05-29 00:00:00


In [13]:
def calculate_cagr(scheme_df, years, latest_date):
    start_date = latest_date - pd.DateOffset(years=years)
    scheme_df = scheme_df[scheme_df['date'] >= start_date]
    if len(scheme_df) < 2:
        return np.nan
    nav_start = scheme_df.iloc[0]['nav']
    nav_end = scheme_df.iloc[-1]['nav']
    actual_start_date = scheme_df.iloc[0]['date']
    actual_end_date = scheme_df.iloc[-1]['date']
    n = (actual_end_date - actual_start_date).days / 365.25
    if n <= 0:
        return np.nan
    cagr = (nav_end / nav_start) ** (1/n) - 1
    return cagr

In [14]:
results = []

for amfi_code, scheme_df in nav_df.groupby('amfi_code'):
    scheme_df = scheme_df.sort_values('date')
    row = {'amfi_code': amfi_code}
    for years in [1, 3, 5]:
        row[f'cagr_{years}yr'] = calculate_cagr(scheme_df, years, latest_date)
    results.append(row)

cagr_df = pd.DataFrame(results)
cagr_df.head(10)

,amfi_code,cagr_1yr,cagr_3yr,cagr_5yr
0,100016,-0.022258,0.012924,0.026371
1,100025,0.037076,0.039155,0.044582
2,100033,0.532772,0.324340,0.301232
3,101206,0.479638,0.289602,0.235384
4,101207,-0.240003,-0.041515,0.079388
5,101208,0.072418,0.063143,0.065090
6,102885,0.202229,0.196624,0.182324
7,102886,-0.168080,-0.007672,0.011717
8,102887,0.135930,0.255497,0.168411
9,118632,0.340079,0.226466,0.240495


In [15]:
cagr_df.isna().sum()

amfi_code    0
cagr_1yr     0
cagr_3yr     0
cagr_5yr     0
dtype: int64

In [16]:
cagr_df.head(10)

,amfi_code,cagr_1yr,cagr_3yr,cagr_5yr
0,100016,-0.022258,0.012924,0.026371
1,100025,0.037076,0.039155,0.044582
2,100033,0.532772,0.324340,0.301232
3,101206,0.479638,0.289602,0.235384
4,101207,-0.240003,-0.041515,0.079388
5,101208,0.072418,0.063143,0.065090
6,102885,0.202229,0.196624,0.182324
7,102886,-0.168080,-0.007672,0.011717
8,102887,0.135930,0.255497,0.168411
9,118632,0.340079,0.226466,0.240495


In [17]:
history_check = nav_df.groupby('amfi_code')['date'].agg(['min', 'max'])
history_check['years_of_data'] = (history_check['max'] - history_check['min']).dt.days / 365.25
cagr_df = cagr_df.merge(history_check['years_of_data'], on='amfi_code')
cagr_df.head(10)

,amfi_code,cagr_1yr,cagr_3yr,cagr_5yr,years_of_data
0,100016,-0.022258,0.012924,0.026371,4.399726
1,100025,0.037076,0.039155,0.044582,4.399726
2,100033,0.532772,0.324340,0.301232,4.399726
3,101206,0.479638,0.289602,0.235384,4.399726
4,101207,-0.240003,-0.041515,0.079388,4.399726
5,101208,0.072418,0.063143,0.065090,4.399726
6,102885,0.202229,0.196624,0.182324,4.399726
7,102886,-0.168080,-0.007672,0.011717,4.399726
8,102887,0.135930,0.255497,0.168411,4.399726
9,118632,0.340079,0.226466,0.240495,4.399726


In [18]:
risk_free_annual = 0.065
risk_free_daily = risk_free_annual / 252
print(risk_free_daily)

0.00025793650793650796


In [19]:
sharpe_results = []

for amfi_code, scheme_df in nav_df.groupby('amfi_code'):
    returns = scheme_df['daily_return'].dropna()
    mean_return = returns.mean()
    std_return = returns.std()
    sharpe = (mean_return - risk_free_daily) / std_return * np.sqrt(252)
    sharpe_results.append({'amfi_code': amfi_code, 'sharpe_ratio': sharpe})

sharpe_df = pd.DataFrame(sharpe_results)
sharpe_df.head(10)

,amfi_code,sharpe_ratio
0,100016,-0.321019
1,100025,-1.039941
2,100033,0.808268
3,101206,0.717409
4,101207,0.052618
5,101208,-4.650401
6,102885,0.519797
7,102886,-0.294889
8,102887,0.384637
9,118632,0.758851


In [20]:
sharpe_df['sharpe_rank'] = sharpe_df['sharpe_ratio'].rank(ascending=False)
sharpe_df = sharpe_df.sort_values('sharpe_rank')
sharpe_df.head(10)

,amfi_code,sharpe_ratio,sharpe_rank
34,148567,1.068224,1.0
30,120843,0.965561,2.0
36,148569,0.919047,3.0
25,120505,0.883256,4.0
19,119551,0.860977,5.0
38,149323,0.832885,6.0
2,100033,0.808268,7.0
9,118632,0.758851,8.0
16,119094,0.730547,9.0
3,101206,0.717409,10.0


In [21]:
sortino_results = []

for amfi_code, scheme_df in nav_df.groupby('amfi_code'):
    returns = scheme_df['daily_return'].dropna()
    mean_return = returns.mean()
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std()
    sortino = (mean_return - risk_free_daily) / downside_std * np.sqrt(252)
    sortino_results.append({'amfi_code': amfi_code, 'sortino_ratio': sortino})

sortino_df = pd.DataFrame(sortino_results)
sortino_df.head(10)

,amfi_code,sortino_ratio
0,100016,-0.472822
1,100025,-1.461220
2,100033,1.144216
3,101206,1.063909
4,101207,0.075668
5,101208,-8.741654
6,102885,0.772972
7,102886,-0.420589
8,102887,0.571858
9,118632,1.098880


In [22]:
import os
os.listdir('../Data/Processed/')

['01_fund_master.csv',
 '03_aum_by_fund_house.csv',
 '05_category_inflows.csv',
 '06_industry_folio_count.csv',
 '09_portfolio_holdings.csv',
 '10_benchmark_indices.csv',
 'investor_transactions_clean.csv',
 'monthly_sip_inflows_clean.csv',
 'nav_history_clean.csv',
 'scheme_performance_clean.csv']